In [1]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
test_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Number of intents:", train_df["category"].nunique())

Train: (10003, 2)
Test: (3080, 2)
Number of intents: 77


In [ ]:
labels = sorted(train_df["category"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print("Number of labels:", len(labels))

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"].tolist(),
    train_df["category"].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=train_df["category"]
)

print("Training examples:", len(train_texts))
print("Validation examples:", len(val_texts))

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=64
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

print("Tokenization complete")

In [ ]:
train_label_ids = [label2id[label] for label in train_labels]
val_label_ids = [label2id[label] for label in val_labels]
test_label_ids = [
    label2id[label] for label in test_df["category"]
]

In [ ]:
import torch
from torch.utils.data import Dataset

class Banking77Dataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = Banking77Dataset(
    train_encodings,
    train_label_ids
)

val_dataset = Banking77Dataset(
    val_encodings,
    val_label_ids
)

test_dataset = Banking77Dataset(
    test_encodings,
    test_label_ids
)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=77,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predicted_ids = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predicted_ids)
    f1 = f1_score(
        labels,
        predicted_ids,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./banking77-bert-improved",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    fp16=False
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("Final Test Accuracy:", test_results["eval_accuracy"])
print("Final Test F1:", test_results["eval_f1"])

In [ ]:
predictions = trainer.predict(test_dataset)

predicted_ids = np.argmax(
    predictions.predictions,
    axis=1
)

predicted_labels = [
    id2label[i]
    for i in predicted_ids
]

true_labels = test_df["category"].tolist()